# Hands-on G²RINS: teaching a computer to read a polymer

**CHEM 73000 · Polymer Chemistry (Part 2) · CUNY · Guest lecture by Dr. Gervasio Zaldivar (de Pablo Lab, NYU)**

<a href="https://colab.research.google.com/github/sloverde/CHEM73000-G2RINS-HandsOn/blob/main/G2RINS_HandsOn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In lecture you saw that a real polymer sample is not *one* molecule. It is a mixture of chains with different lengths and sequences.
**G²RINS** is a short line of text that describes the *whole mixture*, and the `g2rins` Python package can turn that text into
thousands of realistic chains.

In about 30 minutes you will:

| Activity | What you do | Connects to |
|---|---|---|
| 1 | Write polystyrene in G²RINS and draw one chain | Polymer structure |
| 2 | Generate 200 chains and measure M<sub>n</sub>, M<sub>w</sub> and Đ | Lecture 4: molecular weight |
| 3 | Build the EO/PO copolymer from the preprint (Figures 1 and 3) | Chain copolymerization |
| 4 | Make random, blocky and alternating copolymers | Reactivity ratios, block copolymers |

**No programming experience is needed.** You will only run cells and change a few numbers.

## 0 · How to use this notebook (read me first!)

* This page is a **notebook**: a mix of text and gray **code cells**.
* To run a code cell, click it and press **Shift + Enter**, or click the ▶ button at its left.
* Run the cells **in order, from top to bottom**. A green check ✓ appears when a cell has finished.
* If you are asked *"Warning: This notebook was not authored by Google"*, click **Run anyway**.
* You cannot break anything. If something goes wrong, use **Runtime → Restart session** and run the cells again from the top.
* Want to keep your work? Use **File → Save a copy in Drive**.

Try it now: run the cell below. It should print a greeting.

In [ ]:
print("Hello, polymer chemist! Python is working.")

## 1 · Install G²RINS (run once, takes about 1 minute)

This downloads the `g2rins` package from Dr. Zaldivar's group (version 1.0.0) plus the chemistry toolkit RDKit.
You will see a lot of text scroll by. That is normal.

In [ ]:
# Install the G2RINS package (v1.0.0) and the tools used to draw graphs
!pip -q install "g2rins @ git+https://github.com/depablolab/g2rins.git@v1.0.0" pydot
!which dot > /dev/null || apt-get -qq install -y graphviz > /dev/null
print("Installation finished!")

## 2 · Load the helper tools (just run it)

This cell defines a few small helper commands so the rest of the notebook stays simple.
You do **not** need to read or understand it. In Colab it is folded away; just run it.

| Helper | What it does |
|---|---|
| `show_graph(polymer)` | draws the **generative graph** (like Figure 3a in the preprint) |
| `draw_chain(polymer)` | samples **one** chain and draws its chemical structure |
| `make_chains(polymer, n)` | samples **n** chains (an *ensemble*) |
| `mw_report(chains)` | histogram of molecular weights + M<sub>n</sub>, M<sub>w</sub>, Đ |
| `composition(chains)` | how many of each repeat unit were used |
| `sequences(polymer, n, letters)` | writes chains as letter sequences, e.g. `EEPEEEP` |

In [ ]:
# @title Helper tools (click ▶ to run; no need to open) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import SVG, display
from rdkit import Chem, RDLogger
from rdkit.Chem import rdDepictor, Descriptors
from rdkit.Chem.Draw import rdMolDraw2D
import g2rins
from g2rins import StochasticDistribution

RDLogger.DisableLog("rdApp.*")
_rng = np.random.default_rng(2026)
try:
    import pydot
except ImportError:
    pydot = None


def _creator(polymer):
    return g2rins.G2rins.make(polymer).get_graph_creator()


def show_graph(polymer, color_by="unit"):
    """Draw the generative graph. color_by='unit' groups atoms by building block, 'atom' colors by element."""
    dot = _creator(polymer).get_dot_string(color_by=color_by)
    if pydot is None:
        print("Graph drawing is not available (pydot missing).")
        return
    svg = pydot.graph_from_dot_data(dot)[0].create_svg().decode("utf-8")
    display(SVG(svg.replace("svg:", "")))


def draw_chain(polymer, width=650, height=220):
    """Sample ONE chain from the polymer description and draw it."""
    graph = _creator(polymer).get_ensemble_creator().sample_mol_graph(rng=_rng)
    mol = g2rins.mol_graph_to_rdkit_mol(graph)
    mol = Chem.Mol(mol.ToBinary())
    rdDepictor.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    display(SVG(drawer.GetDrawingText().replace("svg:", "")))
    print(f"Molecular weight of this chain: {Descriptors.MolWt(mol):.0f} g/mol")
    print("SMILES of this chain:", Chem.MolToSmiles(mol))


def make_chains(polymer, n=200):
    """Sample n chains. Returns an object with .chains (SMILES), .mol_weights, .units, .distributions."""
    data = _creator(polymer).get_ensemble_creator().create_ensemble(n, output_format="smiles", ensemble_info=True)
    print(f"Made {len(data.chains)} chains. First chain (SMILES):")
    print("  ", data.chains[0][:120] + ("..." if len(data.chains[0]) > 120 else ""))
    return data


def mw_report(data, bins=20):
    """Histogram of chain molecular weights compared with the target distribution, plus Mn, Mw and dispersity."""
    for key in sorted(data.mol_weights):
        target = data.distributions.get(key, "")
        if not target:
            continue
        M = np.asarray(data.mol_weights[key], dtype=float)
        Mn = M.mean()
        Mw = (M ** 2).sum() / M.sum()
        fig, ax = plt.subplots(figsize=(7, 3.8))
        counts, edges, _ = ax.hist(M, bins=bins, color="#1B7F79", alpha=0.65, label=f"{len(M)} generated chains")
        dist = StochasticDistribution.make(target.strip("|"))
        x = np.linspace(max(1.0, 0.7 * M.min()), 1.3 * M.max(), 400)
        ax.plot(x, [dist.prob_mw(float(v)) * len(M) * (edges[1] - edges[0]) for v in x], color="#C97F00", lw=2.5, label=f"target {target.strip('|')}")
        ax.set_xlabel("Molecular weight of a chain (g/mol)")
        ax.set_ylabel("Number of chains")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.show()
        print(f"Mn = {Mn:,.0f} g/mol    Mw = {Mw:,.0f} g/mol    dispersity Đ = Mw/Mn = {Mw / Mn:.3f}")


def composition(data):
    """Count how many times each repeat unit was used in all chains."""
    reps = {k: v for k, v in data.units.items() if k.startswith("R")}
    total = sum(v["frequency"] for v in reps.values())
    for k, v in reps.items():
        print(f"{v['g2rins']:>28} used {v['frequency']:>6} times  ->  {100 * v['frequency'] / total:5.1f} %")


def sequences(polymer, n=5, letters=None):
    """Write n chains as letter sequences of their repeat units (R0, R1, ... or your own letters)."""
    gc = _creator(polymer)
    label = {str(k): v for k, v in g2rins.derive_unit_labels(gc.get_generative_graph()).unit_id.items()}
    data = gc.get_ensemble_creator().create_ensemble(n, output_format="mol_graph", ensemble_info=True)
    letters = letters or {}
    for i, chain in enumerate(data.sequences, 1):
        seq = []
        for part in chain:
            for unit in part:
                uid = next(label[d["origin_idx"]] for _, d in unit.nodes(data=True) if d["atomic_num"] > 0)
                if uid.startswith("R"):
                    seq.append(letters.get(uid, uid))
        print(f"chain {i}: {''.join(seq)}")


print("Helper tools loaded. You are ready to go!")

## Activity 1 · Your first polymer: polystyrene

Here is polystyrene written in G²RINS (this is the *Quickstart* example from the `g2rins` repository):

```
{[] [<]CC([>])c1ccccc1; [>][H]; [<][H] []}|gauss(1000, 120)|
```

Read it piece by piece:

| Piece | Meaning |
|---|---|
| `{ ... }` | a **stochastic object**: "many chains made by one polymerization" |
| `[<]CC([>])c1ccccc1` | the **repeat unit**: styrene, –CH<sub>2</sub>–CH(C<sub>6</sub>H<sub>5</sub>)– written in SMILES. `[<]` and `[>]` are the two places where it bonds to its neighbors |
| `[>][H]` | the **initiator** (chain start): a hydrogen |
| `[<][H]` | the **terminator** (chain end): a hydrogen |
| `\|gauss(1000, 120)\|` | the **molecular-weight distribution**: Gaussian with mean 1000 g/mol and width 120 g/mol |

`[<]` only connects to `[>]`, like a plug and a socket. Run the cells below.

In [ ]:
polystyrene = "{[] [<]CC([>])c1ccccc1; [>][H]; [<][H] []}|gauss(1000, 120)|"

# Draw the generative graph: the 'recipe map' the computer follows to build chains
show_graph(polystyrene)

**How to read the graph.** Each oval is an atom, colored by the building block it belongs to.
* **Black arrows (weight 1.0)** are the fixed bonds *inside* a building block: they always come together.
* **Green arrows** = *transition*: the first step, from the initiator to the first repeat unit.
* **Red arrows** = *propagation*: add another repeat unit. They form a **loop**, which is how the chain grows over and over.
* **Blue arrows** = *termination*: cap the chain end.

This is the same idea as **Figure 3a** of the preprint (the paper uses slightly different colors).
The graph does not decide *how long* a chain is: that comes from the distribution `|gauss(1000, 120)|`.

Now sample **one** real chain from this recipe. Run the cell several times: you get a different chain each time!

In [ ]:
draw_chain(polystyrene)

> **Question 1.1** · The repeat unit of polystyrene has a mass of 104 g/mol. About how many styrene units are in a 1000 g/mol chain?
>
> *Your answer:* (double-click here to type)

## Activity 2 · Many chains at once: M<sub>n</sub>, M<sub>w</sub> and dispersity

A real sample has *billions* of chains. Let's make 200 of them and look at the distribution of chain molecular weights,
exactly as you would measure it by SEC (Lecture 4).

In [ ]:
ps_chains = make_chains(polystyrene, n=200)
mw_report(ps_chains)

Remember from Lecture 4:  M<sub>n</sub> = Σ N<sub>i</sub>M<sub>i</sub> / Σ N<sub>i</sub>,  M<sub>w</sub> = Σ N<sub>i</sub>M<sub>i</sub>² / Σ N<sub>i</sub>M<sub>i</sub>,  Đ = M<sub>w</sub>/M<sub>n</sub> ≥ 1.

### ✏️ Your turn: change the distribution

Edit **only the part between the bars** `|...|` in the next cell and run it. Try each of these:

| Try this distribution | Typical of | Expected Đ |
|---|---|---|
| `log_normal(5000, 1.05)` | living anionic polymerization | ≈ 1.05 (narrow) |
| `schulz_zimm(10000, 5000)` | step-growth / free-radical (M<sub>w</sub>, M<sub>n</sub>) | ≈ 2 (broad) |
| `poisson(3000)` | ideal living chain growth | very close to 1 |

Larger molecular weights take a few more seconds to generate.

In [ ]:
my_distribution = "log_normal(5000, 1.05)"     # <-- change me!

my_polystyrene = "{[] [<]CC([>])c1ccccc1; [>][H]; [<][H] []}|" + my_distribution + "|"
mw_report(make_chains(my_polystyrene, n=200))

> **Question 2.1** · Which of your samples had the largest Đ? Which polymerization mechanism would give that?
>
> **Question 2.2** · Why does the histogram of 200 chains not match the target curve perfectly? What would happen with 2000 chains?
>
> *Your answers:*

## Activity 3 · The EO/PO copolymer from the preprint (Figures 1 and 3)

This is the random copolymer of **ethylene oxide (EO)** and **propylene oxide (PO)** that Dr. Zaldivar used as the first example in the paper:

```
{[] [<|0.8|]CCO[>|0.8|], [<|0.2|]CC(C)O[>|0.2|]; CO[>]; [<][H] []}|log_normal(1400,1.15)|
```

* Two repeat units: `CCO` = EO and `CC(C)O` = PO.
* The numbers `|0.8|` and `|0.2|` are **connection weights**: at every growth step, EO is added with probability 0.8 and PO with 0.2.
* Initiator `CO[>]` = a methoxy group (CH<sub>3</sub>O–), terminator `[<][H]` = H, so chains are CH<sub>3</sub>O–(EO/PO)<sub>n</sub>–H.
* `log_normal(1400, 1.15)`: M<sub>n</sub> = 1400 g/mol and Đ = 1.15.

In [ ]:
eo_po = "{[] [<|0.8|]CCO[>|0.8|], [<|0.2|]CC(C)O[>|0.2|]; CO[>]; [<][H] []}|log_normal(1400,1.15)|"

show_graph(eo_po)      # compare with Figure 3a of the preprint!

Find in the graph: the **initiator** (C–O, top), the **two repeat units** (EO and PO, two shades of purple), the **terminator** (H, bottom), the green start arrows and the red growth loops labeled 0.8 and 0.2.
Now draw a chain and check the composition of a whole ensemble.

In [ ]:
draw_chain(eo_po)

In [ ]:
eo_po_chains = make_chains(eo_po, n=200)
composition(eo_po_chains)
mw_report(eo_po_chains)

### ✏️ Your turn: change the composition

Change the two weights below (they should add up to 1) and see how the composition changes.

In [ ]:
w_EO = 0.5      # <-- change me
w_PO = 0.5      # <-- change me

my_copolymer = f"{{[] [<|{w_EO}|]CCO[>|{w_EO}|], [<|{w_PO}|]CC(C)O[>|{w_PO}|]; CO[>]; [<][H] []}}|log_normal(1400,1.15)|"
print(my_copolymer)
composition(make_chains(my_copolymer, n=200))

> **Question 3.1** · With 80 % EO, what is the average mass of one repeat unit? (EO = 44.05 g/mol, PO = 58.08 g/mol.)
> About how many repeat units does a 1400 g/mol chain contain?
>
> *Your answer:*

## Activity 4 · Random, blocky and alternating copolymers

The *order* of monomers along the chain matters as much as the composition (think of reactivity ratios r<sub>1</sub>, r<sub>2</sub>
from the copolymerization lecture, and of the block copolymers in Lecture 3).
In G²RINS, a bond connector can carry a **list** of weights: the probability of joining to each other connector.
Below, each chain is printed as letters: **E** = ethylene oxide, **P** = propylene oxide.

In [ ]:
letters = {"R0": "E", "R1": "P"}

random_cp = "{[] [<|0.5|]CCO[>|0.5|], [<|0.5|]CC(C)O[>|0.5|]; CO[>]; [<][H] []}|log_normal(1400,1.15)|"
blocky_cp = "{[] [<]CCO[>|0.95 0 0.05 0|], [<]CC(C)O[>|0.05 0 0.95 0|]; CO[>]; [<][H] []}|log_normal(1400,1.15)|"
alternating_cp = "{[] [<]CCO[>|0 0 1 0|], [<]CC(C)O[>|1 0 0 0|]; CO[>]; [<][H] []}|log_normal(1400,1.15)|"

print("RANDOM (like r1*r2 = 1):");       sequences(random_cp, 4, letters)
print("\nBLOCKY (r1, r2 >> 1):");         sequences(blocky_cp, 4, letters)
print("\nALTERNATING (r1 = r2 = 0):");     sequences(alternating_cp, 4, letters)

**What do the lists mean?** In `[>|0.95 0 0.05 0|]` the four numbers are the weights for bonding to the four connectors in the
string, in order: EO's `[<]`, EO's `[>]`, PO's `[<]`, PO's `[>]`. A zero means "cannot bond" (a `>` never bonds to another `>`).
So `0.95 0 0.05 0` attached to EO means "after an EO, add another EO 95 % of the time and a PO 5 % of the time".

> **Question 4.1** · Which of the three samples would you use to make a micelle-forming surfactant, and why? (Lecture 5)
>
> **Question 4.2** · Look at the alternating chains. Do they all start with the same letter? Why or why not?
>
> *Your answers:*

## ⭐ Bonus · A three-armed star polymer

Glycerol has three –OH groups, so it can start **three arms** at once. The arms are a random copolymer of lactic and glycolic acid
(a biodegradable PLGA star, from Section 5.5 of the preprint). Run it a few times: do you always get three arms?

In [ ]:
star = "{[] [<]C(C)C(=O)O[>], [<]CC(=O)O[>]; [>]OCC(O[>])CO[>]; [<][H] []}|schulz_zimm(1800, 1200)|"
show_graph(star)
draw_chain(star)

## Wrap-up

You have used the same software that Dr. Zaldivar's group uses to build simulation boxes and train machine-learning models.
Some questions to think about (and to use in your homework summary):

1. What information does a G²RINS string contain that a single SMILES string cannot?
2. Why is it important that the generated ensemble reproduces the *distribution* (M<sub>n</sub>, Đ), not just the average?
3. How could these ensembles be used to set up a molecular-dynamics simulation (Lecture 5)?

### Credits and further reading

* G²RINS code: <https://github.com/depablolab/g2rins> (GPL-3.0). The full user guide with 12 examples is `G2RINS_guide.ipynb` in that repository.
* Paper: G. Zaldivar, Y. Tian, G. Sun, C. Kappatou, P. Eiden, N. B. Wulkow, V. Settels, J. J. de Pablo,
  *G2RINS: A Generative String-and-Graph Polymer Representation to Assist Computational Materials Discovery*, ChemRxiv (2026),
  [doi:10.26434/chemrxiv.15007504/v1](https://doi.org/10.26434/chemrxiv.15007504/v1)
* Background: Schneider, Walsh, Olsen, de Pablo, *Generative BigSMILES*, Digital Discovery **3**, 51–61 (2024).